<a href="https://colab.research.google.com/github/Jevidha/Applied-AI/blob/main/Ex9%20Trans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers sentencepiece accelerate

In [1]:
import torch
print(torch.cuda.is_available())

True


In [3]:
# ===============================
# 1. Import Libraries
# ===============================
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

# ===============================
# 2. Load Dataset (EN-FR)
# ===============================
dataset = load_dataset("Helsinki-NLP/opus-100", "en-fr")

# Use subset for faster training
dataset = dataset["train"].select(range(20000))

# ===============================
# 3. Load Model & Tokenizer
# ===============================
model_name = "t5-small"   # or "Helsinki-NLP/opus-mt-en-fr"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ===============================
# 4. Preprocessing
# ===============================
def preprocess(example):
    inputs = ["translate English to French: " + ex["en"]
              for ex in example["translation"]]
    targets = [ex["fr"] for ex in example["translation"]]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset = dataset.map(preprocess, batched=True)

# ===============================
# 5. Train-Test Split
# ===============================
dataset = dataset.train_test_split(test_size=0.1)

# ===============================
# 6. Data Collator
# ===============================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ===============================
# 7. Training Arguments
# ===============================
training_args = TrainingArguments(
    output_dir="/content/results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/content/logs",
    fp16=True   # IMPORTANT for T4 GPU
)

# ===============================
# 8. Trainer
# ===============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer   # ✅ NEW replacement for tokenizer
)

# ===============================
# 9. Train Model
# ===============================
trainer.train()

# ===============================
# 10. Save Model
# ===============================
trainer.save_model("/content/translation_model")

# ===============================
# 11. Inference Function
# ===============================
def translate(text):
    input_text = "translate English to French: " + text
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=50,
        num_beams=4   # better translations
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ===============================
# 12. Test Translation
# ===============================
print(translate("How are you?"))
print(translate("Machine learning is interesting"))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.316983,0.284857
2,0.304958,0.283048


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Comment êtes-vous?
L'apprentissage des machines est intéressant
